In [2]:
# Auto-reload modules during development
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

# Import the framework
from nowkit.nowcasting_framework import (
    NowcastConfig,
    DataManager,
    ModelManager,
    EvaluationManager,
    VisualizationManager,
    InferencePipeline
)

# Import model
from xgboost import XGBRegressor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Configuration

In [3]:
# Configure XGBoost experiment
# Using 7 lags and 10-model ensemble
config = NowcastConfig(
    target_variable="gdpc1",
    test_start_date="2005-03-01",
    test_end_date="2010-03-01",
    n_lags=7,  # More lags for gradient boosting
    n_ensemble_models=10,  # Average 10 models due to stochasticity
    quarterly_only=True
)

print("XGBoost Configuration:")
print("=" * 60)
for key, value in config.to_dict().items():
    print(f"{key:20s}: {value}")
print("=" * 60)

XGBoost Configuration:
target              : gdpc1
test_period         : 2005-03-01 to 2010-03-01
lags                : [-2, -1, 0, 1, 2]
n_lags              : 7
n_ensemble          : 10


## 2. Data Loading

In [4]:
# Initialize and load data
data_manager = DataManager(config)
data_manager.load_data().prepare_test_data()

# Display data summary
data_manager.summary()

✓ Loaded data: (913, 25)
✓ Loaded metadata: (24, 8)
✓ Test data prepared: 21 periods


,Metric,Value
0,Total Observations,913
1,Test Periods,21
2,Variables,24
3,Target Variable,gdpc1
4,Missing in Target (%),66.9%


## 3. Model Training and Backtesting

In [ ]:
# Configure XGBoost parameters
xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.1,
    'max_depth': 3,
    'random_state': 42,
    'verbosity': 0
}

# Create model manager and run backtest
xgb_model = ModelManager(XGBRegressor, xgb_params, config)
xgb_model.run_backtest(data_manager)


Running backtest with 10 ensemble model(s)...
  Processing 5/21 dates...
  Processing 5/21 dates...


## 4. Evaluation

In [ ]:
# Evaluate model performance
evaluator = EvaluationManager()
evaluator.add_model_results(
    'XGBoost (10-ensemble)',
    xgb_model.get_predictions(),
    data_manager.actuals,
    config.lags
)

# Display performance metrics
print("\nPERFORMANCE BY VINTAGE:")
print("-" * 60)
display(evaluator.get_performance_table().round(6))

print("\n" + evaluator.summary_report())

## 5. Visualization

In [ ]:
# Create visualization manager
viz = VisualizationManager(evaluator)

# Plot predictions vs actuals
fig1 = viz.plot_predictions_vs_actuals('XGBoost (10-ensemble)')
plt.show()

In [ ]:
# Plot error distribution
fig2 = viz.plot_error_distribution('XGBoost (10-ensemble)')
plt.show()

## 6. Production Inference

In [ ]:
# Create inference pipeline
inference = InferencePipeline(xgb_model, data_manager, config)

# Predict future quarter
prediction = inference.predict_new_date("2010-06-01")

print("\nFUTURE PREDICTION:")
print("=" * 60)
for key, value in prediction.items():
    print(f"{key:20s}: {value}")

## Summary

This notebook implements XGBoost for GDP nowcasting. XGBoost is a gradient boosting implementation that builds an ensemble of weak learners sequentially, with each new tree trying to correct errors from previous ones. We use 1000 trees with a learning rate of 0.1 and average across 10 trained models.

The model uses 7 lags, more than the other approaches, which allows XGBoost to capture more complex temporal patterns. While the training takes longer than simpler models, gradient boosting often achieves strong predictive performance.